In [19]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('null handle').getOrCreate()

In [6]:
from google.colab import files

uploaded = files.upload()

Saving bookings.csv to bookings.csv


In [31]:
from pyspark.sql.functions import max, min, avg, sum, when, col, array_contains

nbooking_df = spark.read.csv(
    'bookings.csv',
    header = True,
    inferSchema = True
)

nbooking_df.printSchema()
nbooking_df.show()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|   

In [32]:
print('total rec: ', nbooking_df.count())
print('city = None')
nbooking_df.filter(col('city').isNull()).show()
print('provider = None')
nbooking_df.filter(col('provider').isNull()).show()
print('booking_amount = None')
nbooking_df.filter(col('booking_amount').isNull()).show()
print('booking_status = None')
nbooking_df.filter(col('booking_status').isNull()).show()
print('payment_mode = None')
nbooking_df.filter(col('payment_mode').isNull()).show()
print("col('None')")
for c_l in nbooking_df.columns:
  print(c_l + ':' , end='')
  print(nbooking_df.filter(col(c_l).isNull()).count())
print('drop null rows')
nbooking_df.na.drop().show()

total rec:  15
city = None
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+

provider = None
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hydera

In [33]:
print('drop booking amount null')
nbooking_df.na.drop(subset=['booking_amount']).show()
print('drop null in [customer_name, service_type, booking_amount]')
nbooking_df.na.drop(subset=['customer_name', 'service_type', 'booking_amount']).show()
print('fill null city : Unknown')
nbooking_df.na.fill({
    'city': 'Unknown'
}).show()
print('fill null provider : Not Available')
nbooking_df.na.fill({
    'provider': 'Not Available'
}).show()
print('fill null payment_mode : Not Provided')
nbooking_df.na.fill({
    'payment_mode': 'Not Provided'
}).show()
print('fill null booking_status : Unknown')
nbooking_df.na.fill({
    'booking_status': 'Unknown'
}).show()
print('fill null booking_amount : 0')
nbooking_df.na.fill({
    'booking_amount': 0
}).show()
print('creating full_df')
full_nbooking_df = nbooking_df.withColumn(
    'data_quality_status',
    when(col('booking_id').isNull() | col('booking_status').isNull() | col('booking_amount').isNull(), 'Incomplete')
    .otherwise('Complete')
)
full_nbooking_df.show()
print('rec count by data_quality_status')
full_nbooking_df.groupby('data_quality_status').count().show()
print('only complete rec')
full_nbooking_df.filter(col('data_quality_status') == 'Complete').show()


drop booking amount null
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL

In [34]:
print('only Incomplete rec')
full_nbooking_df.filter(col('data_quality_status') == 'Incomplete').show()
print('create tax col')
full_nbooking_df = full_nbooking_df.withColumn(
    'tax',
    col('booking_amount') * 0.05
)
print('create final amount col')
full_nbooking_df = full_nbooking_df.withColumn(
    'final_amount',
    col('booking_amount') + col('tax')
)
full_nbooking_df.show()
print('revenue from confirmed books only')
full_nbooking_df.filter(col('booking_status')== 'Confirmed').select(sum(col('final_amount')).alias('total revenue')).show()
print('count by service_type')
full_nbooking_df.groupBy('service_type').count().show()
print('count by city')
full_nbooking_df.groupby('city').count().show()
print('avg amount')
full_nbooking_df = full_nbooking_df.na.fill({
    'booking_amount': 0
})
full_avg = full_nbooking_df.select(avg(col('booking_amount')).alias('avg booking amount'))
full_avg.show()
print('null drop avg amount')
avg = nbooking_df.na.drop(subset=['booking_amount']).select(avg(col('booking_amount')).alias('avg booking amount'))
avg.show()
full_avg_value = full_avg.collect()[0]['avg booking amount']
avg_value = avg.collect()[0]['avg booking amount']

print('avg compare: ', avg_value - full_avg_value)

only Incomplete rec
+----------+-------------+------+------------+-------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|  city|service_type|     provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+-------------------+
|      1005|   Vikram Rao|Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|         Incomplete|
|      1007|    Imran Ali|  Pune|       Hotel|   Budget Inn|          2200|          NULL|         UPI|         Incomplete|
|      1014|   Kavya Nair|Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|         Incomplete|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+-------------------+

create tax col
create final amount col
+----------+-------------+---------+------------+----------------+------

In [24]:
full_nbooking_df.write.mode('overwrite').parquet('clean_bookings.parquet')

In [30]:
full_nbooking_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|      6825.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|      4725.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|     12600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|    

In [35]:
uploaded02 = files.upload()

Saving customers.json to customers.json


In [36]:
customers_df = spark.read.option(
    'multiline',
    'true'
).json('customers.json')

customers_df.printSchema()
customers_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |


In [37]:
flat_customers_df = customers_df.select(
  "customer_id",
  "name",
  "city",
  "membership",
      col("contact.phone").alias("phone"),
      col("contact.email").alias("email"),
      col("preferences.preferred_service").alias("preferred_service"),
      col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [40]:
flat_customers_df.select('name', 'city', 'email').show()
flat_customers_df.filter(col('city').isNull()).show()
flat_customers_df.filter(col('phone').isNull()).show()
flat_customers_df.filter(col('email').isNull()).show()
flat_customers_df.filter(col('membership').isNull()).show()
flat_customers_df.filter(col('preferred_service').isNull()).show()
flat_customers_df.filter(col('budget_range').isNull()).show()
print('null by col')
for c in flat_customers_df.columns:
  print(c, end=" : ")
  print(flat_customers_df.filter(col(c).isNull()).count())
full_flat_customers_df = flat_customers_df.na.fill({
    'city': 'Unknown'
})
full_flat_customers_df = full_flat_customers_df.na.fill({
    'membership': 'Standard',
    'phone': 'Not Provided',
    'email': 'Not Provided',
    'preferred_service': 'Not Selected',
    'budget_range': 'Unknown'
})
full_flat_customers_df.show()

full_flat_customers_df = flat_customers_df.withColumn(
    'customer_quality_status',
    when(col('city').isNull() | col('email').isNull() | col('phone').isNull() | col('membership').isNull() | col('preferred_service').isNull(), 'Incomplete')
    .otherwise('Complete')
)
full_flat_customers_df.show()

+------------+---------+---------------+
|        name|     city|          email|
+------------+---------+---------------+
| Aarav Mehta|Hyderabad| aarav@mail.com|
|   Sana Khan|Bangalore|  sana@mail.com|
| John Mathew|     NULL|           NULL|
|Ayesha Begum|Hyderabad|ayesha@mail.com|
|  Vikram Rao|   Mumbai|           NULL|
+------------+---------+---------------+

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service

In [45]:
full_flat_customers_df.groupBy('customer_quality_status').count().show()
full_flat_customers_df.filter(col('customer_quality_status') == 'Complete').show()
full_flat_customers_df.filter(col('customer_quality_status') == 'Incomplete').show()
full_flat_customers_df.groupby('membership').count().show()
full_flat_customers_df.groupby('preferred_service').count().show()
full_flat_customers_df.write.mode('overwrite').parquet('customers_flat.parquet')
full_flat_customers_df.write.mode('overwrite').csv('clean_customers.csv')
print('diff rec count : ', end="")
print(customers_df.count() - full_flat_customers_df.count())
customers_df.filter(col('contact.phone').isNull() & col('contact.email').isNull()).show()
customers_df.filter(
    col('preferences.budget_range').isNull() |
    col('preferences.preferred_service').isNull()
).show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|     phone|         email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|          1|Aarav Mehta|Hyderabad|      Gold|9876500011|aarav@mail.com|           Flight|      Medium|               Complete|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|